# NDVI Calculation

We are going to work with images from the **Sentinel-2** satellite, part of the Copernicus programme. Sentinel-2 records the light reflected by the Earth's surface in 13 spectral **bands**, and not all of them have the same resolution:

| Resolution | Bands |
|---|---|
| 10 m | B02 (blue), B03 (green), **B04 (red)**, **B08 (near infrared)** |
| 20 m | B05, B06 and B07 (red edge), B8A (narrow near infrared), B11 and B12 (shortwave infrared) |
| 60 m | B01 (aerosols), B09 (water vapour), B10 (cirrus; only in the L1C product) |

In this exercise you will visualize the **Red (B04)** and **Near Infrared (B08)** bands and, from them, calculate and analyze the **NDVI (Normalized Difference Vegetation Index)**, an index widely used to study the condition of vegetation. Both bands have the satellite's finest resolution: if downloaded at their original resolution, each pixel covers a 10 m x 10 m square, that is, **100 m²** of surface. When downloading you can choose a lower resolution (larger pixels) so that the files take up less space.

Before the workshop, complete the first section (**Setup**).

## Setup

1. Install the dependencies following the **Install Dependencies** section of the repository's `README.md`: it explains how to create a Python environment and install the packages in `requirements.txt`.
2. Open this notebook with that environment and run the next cell. If it prints the library versions without errors, the setup is ready.

The images we are going to use already come in the repository's `class/raw/` folder.


In [ ]:
# Import libraries
import rasterio
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import re

# Show the library versions to check that they are installed
print(f"rasterio version: {rasterio.__version__}")
print(f"numpy version: {np.__version__}")

Documentation, examples and explanations for each library you are going to use:

- [**rasterio**](https://rasterio.readthedocs.io/): To read and process raster data.
- [**numpy**](https://numpy.org/doc/): For efficient numerical operations.
- [**pathlib**](https://docs.python.org/3/library/pathlib.html): To handle file paths in a convenient, modern way.
- [**matplotlib**](https://matplotlib.org/stable/contents.html): To visualize data and images.
- [**re (Regular expressions)**](https://docs.python.org/3/library/re.html): To work with text patterns.

## Step-by-Step Guide for Geospatial Analysis

### Step 1: Accessing Sentinel images

- The **[SentiWiki](https://sentiwiki.copernicus.eu/web/sentiwiki)** has information about the satellites, their design and the data they acquire.
- Access to Sentinel data: [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/).
- To search for and download images from the web browser: [Copernicus Browser](https://browser.dataspace.copernicus.eu/).

The images for this exercise are already downloaded in `class/raw/`. In this step and the next one you will see where they come from and how to get them: you will need it at the end, to repeat the analysis with your own images.

To download an image you define the area of interest (AOI) with coordinates (latitude and longitude) and choose the bands you need.

**What are you actually opening? A raster file.**

A satellite image can be thought of as a grid (matrix) where each pixel represents a small piece of the Earth's surface and stores a reflectance value in a given band of the spectrum (Red, NIR, Blue, Green, etc.).

`.tiff` (GeoTIFF) files usually also include important metadata:
- Coordinate reference system.
- Spatial resolution.
- Number of bands.
- Additional information about the sensor and the product.

The idea is that you understand what you are handling before throwing operations at it.

> If all this sounds like a lot of theory, keep the basic idea: "each pixel is a little number", and we will add details as we go.


### Step 2: Downloading bands

Identify and download the bands needed to calculate the **NDVI**. You can save your images next to the existing ones, in the same folder (`class/raw/ndvi/`). In Sentinel-2 these are:

- The **Red** band (B04).
- The **NIR** band (Near Infrared, B08).

When downloading:

- Define the exact location using coordinates (lat/long) or a polygon.
- In the Copernicus Browser, use the **Analytical** download and choose bands **B04** and **B08** in their **Raw** version (Sentinel-2 L2A).
- Choose the **TIFF** format with a **16-bit** depth: values range from 0 to 65,535 (reflectance × 65,535).
- Choose the resolution. The images in `class/raw/ndvi/` were downloaded with the *MEDIUM* option: their pixels are about **16 m** instead of the original 10 m, and the files are smaller. That is enough for this exercise.
- You can use `pathlib` to handle paths and move/copy files in an orderly way.

**What is a pixel ("px")?**

It is the smallest unit of the image. Each pixel covers an area on the ground (for example, 10 m x 10 m for some Sentinel-2 bands) and stores a reflectance value.

**Reflectance:** the fraction of the sunlight reaching an area that the surface reflects. It is 0 if it reflects nothing and 1 if it reflects everything.

When you use `rasterio.open(...)`, look at attributes such as:

- `type(src).__name__`
- `src.width`, `src.height`
- `src.crs` (coordinate system)
- `src.count` (number of bands)
- `src.meta` (dictionary with metadata)
- the shape of `src.read()` and `src.read(1)`


In [ ]:
# Sentinel-2 L2A bands downloaded for the exercise
directory_path = Path('raw/ndvi')

# Names of the TIFF files in the folder
for file_path in directory_path.glob("*.tiff"):
    print(file_path)
print('-' * 20)

# Pick one file to explore, for example the red band (B04)
file_path = next(directory_path.glob("*B04*.tiff"))
print('Chosen file:', file_path.name)
print('*' * 20)

# ...........................
# Write the solution below:
# ...........................

### Step 3: Explore, Characterize and Visualize the Data

First you will define a function that reads a band from a `.tiff` file and returns:

- The band data as a `numpy` array.
- The associated metadata (for example, the `rasterio` profile).

Use **rasterio**, a library designed to work with geospatial rasters, and combine it with **numpy** for numerical operations. Convert the data to `float32`, since you will need it for later calculations.

In [ ]:
def read_band(filename):
    """
    Reads the first band of a TIFF file.
    - Parameters:
        filename (Path or str): full path to the file.
    - Returns: tuple (band_data, metadata)
        band_data (numpy.ndarray): band values, converted to float32.
        metadata (dict): file metadata.
    """

    #............................
    # Write the solution below:
    #............................

Once the function is implemented:

1. Test it by reading a band.
2. Inspect the result using:
   - `.min()`
   - `.max()`
   - `.shape`

This gives you an idea of the range of values and the size of the image.

The `date` variable chooses which image to use: this way bands B04 and B08 always come from the same image. If you saved your own images in the folder, change it to the date of yours (it appears at the beginning of each file name).

In [ ]:
directory_path = Path('raw/ndvi')

# Date of the image to analyze (it appears at the beginning of each file name)
date = "2024-11-22"

for file_path in sorted(directory_path.glob(f"{date}*.tiff")):
    print(file_path.name)

    if "B08" in file_path.name:  # the file is band B08 (NIR)
        B08, _ = read_band(file_path)
        #..........................
        # Complete the code below:
        #..........................

    elif "B04" in file_path.name:  # the file is band B04 (red)
        B04, _ = read_band(file_path)
        #..........................
        # Complete the code below:
        #..........................

### Step 4: What value does each pixel store?

- Sentinel-2 measures **reflectance**: the fraction of light reflected by the surface, between 0 (none) and 1 (all of it).
- The 16-bit TIFFs we downloaded (Step 2) scale it to the full 16-bit range, from 0 to 65,535 ($=2^{16}-1$).

For example, a value of 20,683 in band B08 corresponds to a reflectance of 20,683 / 65,535 ≈ 0.32.

For the first visualization we use a **fixed scale**, with all the possible values of the file:
- `min_sentinel = 0`
- `max_sentinel = 65535`

This way both bands are shown with the same scale and their colors can be compared with each other.

Visualize the bands. You can use `plt.imshow(...)` with a colormap such as `cmap="Reds_r"`, `cmap="magma"` or any other you like. Show the images side by side (`plt.subplot(num_rows, num_cols, position)`), turn on the axes (`plt.axis("on")` if needed) and try to interpret what you see in each band.

> The axes of these maps show **pixel rows and columns**, not geographic coordinates. To see latitude and longitude you would need the file bounds (`src.bounds`), which you explored in Step 2. In Step 9 we will use the file's georeferencing to locate a pixel on the map and look it up in Google Maps.


In [ ]:
# Possible values of the 16-bit TIFFs (reflectance rescaled from 0 to 65,535)
min_sentinel = 0
max_sentinel = 65535

plt.figure(figsize=(12, 4))

# Visualize both bands side by side
plt.subplot(1, 2, 1)
# Red band (B04)
#..........................
# Complete the code below:
#..........................

plt.subplot(1, 2, 2)
# Near infrared band (B08)
#..........................
# Complete the code below:
#..........................


plt.tight_layout()
plt.show()

#### Visualization scales

With the fixed scale, the colors are spread over all possible values (0 to 65,535). But what we want to study takes up a small part of that range: vegetation reflects:
- only 3–10 % of red light, and
- 20–45 % of near infrared.

That is why almost the whole image ends up in a few shades and details are hard to tell apart. To make better use of the colors, the scale has to be fitted to the values the image actually has:

1. **Min–max:** the scale goes from the lowest to the highest value of each band (this is what `plt.imshow` does if we don't give it `vmin` and `vmax`). It helps somewhat, but the extremes are set by the surfaces that reflect the least and the most: **water** and shadows at the bottom, **clouds** and sand at the top. Vegetation is still concentrated in a narrow range.


In [ ]:
# Both bands with a min–max scale
plt.figure(figsize=(12, 4))

for i, (name, band,cmap) in enumerate([("B04 (red)", B04, "Reds_r"), ("B08 (near infrared)", B08, "magma")], start=1):
    plt.subplot(1, 2, i)
    plt.imshow(band, cmap=cmap)   # no vmin or vmax: min–max scale
    plt.colorbar(label="Reflectance × 65,535")
    plt.title(f"{name}: min–max")

plt.tight_layout()
plt.show()

2. **Percentile stretch:** the scale goes from a lower to an upper percentile of the band. Pixels below the lower one or above the upper one, the **extremes**, are saturated at the minimum or maximum color, and all the colors are spread over the values in between. This is the usual practice in remote sensing.

In both cases each band ends up with **its own scale**: even if we use the same percentiles, each band has its own distribution, so the limits fall on different values. For example, with percentiles 2–98 the lightest color corresponds to about 23,600 in B04 and about 32,200 in B08. That is why, unlike the fixed scale, the same color no longer indicates the same value across bands or images: it only indicates the pixel's position within its own band.

A first example is a **symmetric** clip, from percentile 2 to 98: the lowest 2 % and the highest 2 % of the values of each band are discarded. The result is shown in the **first two images** of the output of the next cell.

To choose the limits better, the **histogram** is a very useful tool: it shows how many pixels there are for each value. The **histograms** in the output of the next cell show which surfaces are present in each band. With percentiles we **"zoom" into the region of values we are interested in** and saturate the rest.

As a reference, the tables for each band show the range of values of each surface type in this image. These values were **calculated separately**, by identifying which pixels correspond to water, vegetation and the rest of the land; here we use them only to read the histogram and choose where to "zoom".

- **Red band (B04).** There is a tall, narrow peak at low values and a long tail to the right. The peak contains both **vegetation** and **water**: both reflect very little red and their values overlap, so **the red band alone cannot separate them**. The tail corresponds to sand, the city and clouds.

  **Visualization goal:** highlight the areas **without vegetation** (soil, fields, city) and their nuances. To do this it is best to **saturate** the water and vegetation peak in the darkest color, with a lower percentile to the right of that peak, and saturate the tail of clouds and sand in the lightest color, with an upper percentile that cuts it off.


| Surface | Typical range in B04 (0 – 65,535 scale) | Median |
|---|---|---|
| Water | 2,600 – 4,500 | 3,300 |
| Vegetation | 2,400 – 8,500 | 4,600 |
| Rest of the land (soil, fields, city) | 4,300 – 15,400 | 9,700 |

*Typical range:* the range containing 90 % of the pixels of each surface (the lowest 5 % and the highest 5 % are left out).



- **Near infrared (B08).** There are two well-separated peaks: a narrow one near 0, **water**, which barely reflects infrared, and a wide "mountain", the land with **vegetation**, which reflects a lot. Unlike red, **infrared does separate water from vegetation**.

  **Visualization goal:** highlight the differences **within vegetation**. To do this it is best to **saturate** water (and land with little vegetation) with a lower percentile above the water peak, and saturate clouds with an upper percentile.

| Surface | Typical range in B08 (0 – 65,535 scale) | Median |
|---|---|---|
| Water | 1,800 – 3,100 | 2,300 |
| Vegetation | 13,600 – 29,300 | 23,000 |
| Rest of the land (soil, fields, city) | 3,700 – 25,300 | 16,200 |

*Typical range:* the range containing 90 % of the pixels of each surface (the lowest 5 % and the highest 5 % are left out).



**Exercise:** in the next cell, change the percentiles of each band in the `chosen` dictionary and compare them with its histogram until the image highlights what we are looking for in that band. The **last two images** show a possible solution, chosen from the histograms:

- **B04, percentiles 55–90:** the lower one (≈ 5,200) falls to the right of the peak and saturates almost all the water and two thirds of the vegetation; the upper one (≈ 13,000) cuts the tail and saturates the clouds, but also the brightest areas of the city and the sand. In exchange, the scale is narrower and soils and fields show more nuances.
- **B08, percentiles 25–95:** the lower one (≈ 12,500) falls between water and vegetation and saturates water and land with little vegetation; the upper one (≈ 29,700) saturates the brightest clouds.

| | B04: percentiles 55–90 | B08: percentiles 25–95 |
|---|---|---|
| Scale | 5,200 – 13,000 | 12,500 – 29,700 |
| Below (saturated in the darkest color) | water 99 %, vegetation 65 %, rest of the land 13 % | water 100 %, rest of the land 40 %, vegetation 3 % |
| Above (saturated in the lightest color) | clouds 100 %, rest of the land 18 % | clouds 40 %, vegetation 4 % |
| Within the scale | rest of the land 69 %, vegetation 35 % | vegetation 93 %, clouds 60 %, rest of the land 59 % |

*The percentages refer to the pixels of each surface: for example, "water 99 %" means that 99 % of the water pixels are saturated.*

These limits are only for **visualizing** each band better; they do not change the data.

The important conclusion is that **each band provides different information**: red distinguishes land without vegetation (soil, fields, city) and infrared separates water from vegetation. Combined, they are the basis of the **NDVI**, the index we calculate in Step 6. Before that, in the next cells, we analyze each band separately and the physics that explains why each surface reflects light the way it does.


In [ ]:
# Percentile stretch: symmetric example, histograms and chosen percentiles for each band
bands = [("B04 (red)", B04, "tab:red", "Reds_r"), ("B08 (near infrared)", B08, "tab:purple", "magma")]
label = "Reflectance × 65,535"

# Percentiles chosen for each band from its histogram
chosen = {"B04 (red)": (55, 90), "B08 (near infrared)": (25, 95)}

# 1) Symmetric example: percentiles 2–98
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, band, _, cmap) in zip(axs, bands):
    lo, hi = np.percentile(band, [2, 98])   # pixels below lo and above hi are saturated
    im = ax.imshow(band, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(f"{name}: percentiles 2–98")
    fig.colorbar(im, ax=ax, label=label)
plt.tight_layout()
plt.show()

# 2) Histograms: distribution of values of each band and where the percentiles cut
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, band, color, _) in zip(axs, bands):
    ax.hist(band.ravel(), bins=200, color=color, alpha=0.7)
    for (p_lo, p_hi), style in [((2, 98), "--"), (chosen[name], ":")]:
        lo, hi = np.percentile(band, [p_lo, p_hi])
        ax.axvline(lo, color="black", linestyle=style, label=f"percentiles {p_lo}–{p_hi}")
        ax.axvline(hi, color="black", linestyle=style)
    ax.set_title(name)
    ax.set_xlabel(label)
    ax.set_ylabel("Number of pixels")
    ax.legend()
plt.tight_layout()
plt.show()

# 3) Percentiles chosen for each band
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, band, _, cmap) in zip(axs, bands):
    p_lo, p_hi = chosen[name]
    lo, hi = np.percentile(band, [p_lo, p_hi])
    im = ax.imshow(band, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(f"{name}: percentiles {p_lo}–{p_hi}")
    fig.colorbar(im, ax=ax, label=label)
plt.tight_layout()
plt.show()

### Step 5: What are the RED and NIR bands?

In remote sensing, **RED** and **NIR** are two specific regions of the electromagnetic spectrum (i.e., of light) captured by sensors on satellites (or drones):

- **The Red band (RED)** measures reflectance in the visible red light spectrum (*wavelength of 600–700 nm*; in Sentinel-2, band B04 covers approximately 650 to 680 nm). Vegetation absorbs most of this red light for photosynthesis (**lower values**), which makes it a key indicator of plant condition. In other words, the amount of reflected red light is inversely related to the amount of chlorophyll in the plants. Bare soil or non-vegetated areas reflect more red light (**higher values**). Look at the sand and the water in the previous images: do they reflect a lot or a little red? Why?

- **The Near Infrared band (NIR)** captures the region of light just beyond the visible spectrum (*wavelength of 700–1000 nm*; in Sentinel-2, band B08 covers approximately 785 to 900 nm). Healthy vegetation reflects a large amount of NIR light for two reasons: it does not use it for photosynthesis and, in addition, the internal structure of the leaves (their cell walls) scatters it and sends it back out. This high reflectance (**higher values**) in the NIR band is characteristic of dense, healthy vegetation. Is there any other component in the image that we have not mentioned? What is it and how does it look in each band?

**Electromagnetic spectrum:**

![Light spectrum](images/Light_Spectrum.png)

<details>
<summary><b>Show answers</b> (try to answer before opening)</summary>

**1. Do sand and water reflect a lot or a little red?**

- **Sand:** reflects **a lot** of red. The strip of beach is among the brightest things in the red band, together with the clouds: dry sand has no chlorophyll to absorb red and reflects all visible light well. It also reflects quite a lot of infrared.
- **Water:** reflects **little** red. It absorbs much of the light reaching it, so in the red band it looks dark and gets confused with vegetation: both end up together in the histogram peak. In the infrared it absorbs almost everything and is the darkest thing in B08; that is why this band does separate it from vegetation.

**2. Is there any other component we did not mention?**

Yes: the **clouds**, to the west of the image. They are very bright in both bands, because they reflect almost all the light at all these wavelengths. That is why they set the maximum of the min–max scale and form the right tail of the histograms.

</details>


### Step 6: NDVI Calculation

The **NDVI (Normalized Difference Vegetation Index)** is an index widely used in remote sensing to analyze vegetation cover and condition.

It is defined from the reflectance in the **Red (RED)** band and in the **Near Infrared (NIR)** band. In simple terms:

- Healthy vegetation absorbs a lot of red light and reflects quite a lot of NIR.
- Surfaces without vegetation (bare soil, buildings) tend to reflect more red and less NIR.

#### Formula

The NDVI is calculated as:

$$\text{NDVI} = \frac{\text{NIR} - \text{RED}}{\text{NIR} + \text{RED}} \qquad\qquad(1)$$

The NDVI always gives a value between -1 and 1. In practice, most values fall within a smaller range: in this image, water gives between -0.3 and -0.06 and vegetation between 0.44 and 0.77.


**Question: do the values of each band need to be converted to reflectance?**

<details>
<summary><b>Show answer</b> (try to answer before opening)</summary>

No. The TIFFs store reflectance multiplied by 65,535 (Step 4), and that factor appears both in the numerator and in the denominator, so it cancels out:

$$\frac{65{,}535\,\text{NIR} - 65{,}535\,\text{RED}}{65{,}535\,\text{NIR} + 65{,}535\,\text{RED}} = \frac{65{,}535\,(\text{NIR} - \text{RED})}{65{,}535\,(\text{NIR} + \text{RED})} = \frac{\text{NIR} - \text{RED}}{\text{NIR} + \text{RED}}$$

That is why we can use bands `B04` and `B08` directly, as we read them. The NDVI gives the same value as with reflectance.

</details>



#### Activities

With the bands you have already loaded and explored:

1. **Avoid division by zero:** make sure that $\text{NIR} + \text{RED} \neq 0$ before applying the formula.
2. **Calculate the NDVI:** use `numpy` arrays to implement formula (1).
3. **Characterize the NDVI:** print the minimum and maximum values and the `shape` of the NDVI array.
4. **Visualize the NDVI:** make a figure to inspect the spatial pattern of the index.
5. Try visualizing it **with** and **without** fixing the scale using `vmin=-1, vmax=1`. What changes in the interpretation of the map?

> If your map comes out all red or all green, it is probably not the satellite's fault: check your code before cursing the hardware.


In [ ]:
# Check whether NIR + RED is 0 in any pixel
# .............................
# Complete the missing code:
# .............................
sum_nir_red = 'COMPLETE THE CODE HERE' # sum of the two bands

# .............................
# Complete the missing code:
# .............................
zeros_exist = 'COMPLETE THE CODE HERE' # Check with a boolean whether any value is 0

print(zeros_exist)

In [ ]:
# Calculate and visualize the NDVI
# .............................
# Complete the missing code:
# .............................

**Question: what changes in the interpretation of the map when the scale is fixed?**

<details>
<summary><b>Show answer</b> (try to answer before opening)</summary>

- **Without fixing the scale (min–max):** the colors go from the minimum of this image (≈ −0.43) to its maximum (≈ 0.91). Yellow, the center of the colormap, falls at ≈ 0.24 instead of 0. That is why water, with a barely negative NDVI (between −0.3 and −0.06), appears in intense red, as if it were very different from the rest, and many nuances can be seen within the vegetation.
- **Fixed scale from −1 to 1:** yellow falls at 0, the boundary between surfaces that reflect more red than infrared (negative NDVI) and those that reflect more infrared than red (positive NDVI). Water looks light orange, because its NDVI is close to 0, and vegetation (NDVI between 0.44 and 0.77) looks green.

With the fixed scale, a color always indicates the same NDVI value, so maps from different dates or places can be compared. With min–max, on the other hand, the colors depend on the extremes of each image: it is useful to see details within a single image, but not to compare images. It is the same thing we saw with the bands in Step 4.

</details>


#### Note on metrics and interpretation (for later)

Later in the notebook you will work with NDVI time series (several years). There you will:

- Calculate different metrics for each image (mean, median, percentage of high vegetation, etc.).
- Compile those metrics into a small dataset.
- Plot the evolution over time with a line chart (X axis: years, Y axis: NDVI metrics).

The key questions will be things like:

- How did the vegetation change between 2016 and 2024?
- Is there recovery, degradation or seasonal patterns?
- What do the metrics say about the "health" of the area?

> You can skip this note for now and come back when you reach the time series part. No need to jump ahead… unless you are one of those people who read the end of the book first.


## Step 7: Additional Indices

Besides the NDVI, other indices can be built from combinations of different bands. One widely used to monitor water bodies is the **NDWI (Normalized Difference Water Index)**.

The NDWI aims to highlight water areas and attenuate vegetation and soil. It is calculated using the **Green (B03 - 559 nm)** band and the **NIR (B08)** band:

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}} \qquad\qquad(2)$$

In this formula, water usually has:
- Low reflectance in NIR (B08).
- Higher reflectance in the green band (B03).

NDWI values also range, in theory, from -1 to 1. Positive values tend to indicate the presence of water; negative values usually correspond to soil, vegetation or urban areas.

**4 exercises:**

1. Download bands **B03** and **B08** (Raw, as in Step 2) to monitor the water of a reservoir anywhere in the world (as an example, I attach those of the Pantà de la Llosa del Cavall, in Sant Llorenç de Morunys, Spain, in `raw/ndwi/`). Choose (at least) two dates on which a noticeable difference in water level can reasonably be expected, such as a winter image and a summer image. If you use your own images, save them in a folder of your own (`raw/ndwi_mine/`) and change `directory_path` in the code.

2. Initialize a dictionary called `ndwi_bands`, reuse the `read_band` function (Step 3) to read each band and complete the code to print the `shape`, the minimum, the maximum and the median of each band for both years.

In [ ]:
directory_path = Path('raw/ndwi')
# Load bands B03 and B08 of each date
# .............................
# Complete the missing code:
# .............................
# {date: {"B03": array, "B08": array}}

for file_path in sorted(directory_path.glob("*_(Raw).tiff")):
    date = re.search(r"\d{4}-\d{2}-\d{2}", file_path.name).group()
    band_name = re.search(r"B\d{2}", file_path.name).group()

    # .............................
    # Complete the missing code:
    # .............................
    data, meta = 'COMPLETE THE CODE HERE'

    ndwi_bands.setdefault(date, {})[band_name] = data

    print(file_path.name)
    # .............................
    # Complete the missing code:
    # .............................
    print(f' - Shape: {"COMPLETE THE CODE HERE"} | Original type: {meta["dtype"]} | CRS: {meta["crs"]}')
    print(f' - Minimum: {"COMPLETE THE CODE HERE"} | Maximum: {"COMPLETE THE CODE HERE"} | Median: {"COMPLETE THE CODE HERE"}'
          f' (reflectance ≈ {np.median(data) / 65535:.2f})')
    print('')


3. Calculate the NDWI of each date with formula (2) (as in Step 6, watch out for divisions by zero) and show, in a grid of 3 rows and 2 columns, bands B03 and B08 and the NDWI, with one column per date. Use an appropriate colormap for each row (for example, `'Greens_r'`, `'magma'` and `'RdBu'` or `'coolwarm'`) and the same scale for both dates, so they can be compared. The common scale of each band is already calculated in the `scales` dictionary.

4.	Visually compare the NDWI of both dates and identify the differences in the state of the reservoir.

In [ ]:
# B03, B08 and NDWI of each date: one column per date
dates = sorted(ndwi_bands)

# Scale common to both dates for each band: percentiles 2–98 of all its pixels
scales = {}
for band_name in ["B03", "B08"]:
    values = np.concatenate([ndwi_bands[d][band_name].ravel() for d in dates])
    scales[band_name] = np.percentile(values, [2, 98])

fig, axs = plt.subplots(3, 2, figsize=(12, 13))

for col, date in enumerate(dates):
    # .............................
    # Complete the missing code:
    # .............................
    B03, B08 = 'COMPLETE THE CODE HERE'
    ndwi = 'COMPLETE THE CODE HERE'

    rows = [
        (B03, "B03 (green)", "Greens_r", *scales["B03"]),
        (B08, "B08 (near infrared)", "magma", *scales["B08"]),
        (ndwi, "NDWI", "RdBu", -1, 1),   # possible NDWI range; blue: water (NDWI > 0)
    ]
    for row, (img, name, cmap, vmin, vmax) in enumerate(rows):
        ax = axs[row, col]
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)   # same scale for both dates
        ax.set_title(f"{name}: {date}")
        ax.axis("off")
        fig.colorbar(im, ax=ax, label="NDWI" if name == "NDWI" else "Reflectance × 65,535", shrink=0.8)

plt.tight_layout()
plt.show()

### Step 8: Operations Over Images

#### NDWI thresholding to isolate water bodies

With the NDWI calculated, you can identify the pixels that correspond to water by applying a **threshold**.

1. Use code similar to the one provided to understand how **boolean masks** work. For example, if you have an array `[0.15, 0.05, 0.30, 0.20]` and choose `threshold = 0.15`, the comparison `array > threshold` gives `[False, False, True, True]`.

In [ ]:
some_array = np.array([0.05, 0.15, 0.15, 0.20, 0.5, 0.75, 1, 1, 1.25, 1.5, 2, 2])
# .............................
# Complete the missing code:
# .............................
my_threshold = 'COMPLETE THE CODE HERE'
masked_array = 'COMPLETE THE CODE HERE'   # True where the value exceeds the threshold

# Histogram: the mask separates the values on each side of the threshold
bins = np.arange(0, 2.2, 0.1)   # bins of 0.1
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(some_array[~masked_array], bins=bins, color="tab:red", label="False: does not exceed the threshold")
ax.hist(some_array[masked_array], bins=bins, color="tab:blue", label="True: exceeds the threshold")
ax.axvline(my_threshold, color="black", linestyle="--", label=f"threshold = {my_threshold}")
ax.set_xlabel("Value")
ax.set_ylabel("Number of values")
ax.legend()
plt.show()

2. Implement a function that, given an NDWI and a threshold, returns a binary mask (water / no water).

In [ ]:
def apply_mask(data, threshold=0):
    """
    Creates a boolean mask from an index (for example, NDWI or NDVI).
    - Parameters:
        data (numpy.ndarray): index values.
        threshold (float): value above which a pixel is marked as True
            (for example, water in the NDWI or vegetation in the NDVI).
    - Returns:
        boolean numpy.ndarray: True where data > threshold.
    """
    # .............................
    # Complete the missing code:
    # .............................

3. Analyze the range of NDWI values (minimum, maximum, histogram). The **histogram** is an excellent tool to tune the threshold: it shows how the values of land and water pixels are distributed, and where it is best to cut to separate them.
4. Visualize how the image changes when the mask is applied, in a 3x2 grid: in row 1, the NDWI of dates 1 and 2; in rows 2 and 3, the NDWI masked with each threshold. You have **two thresholds to play with** (`threshold_1` and `threshold_2`): change them and watch which pixels enter or leave the mask.

Key questions:

- What happens when you lower or raise the threshold value?
- Are there misclassified areas (for example, areas that are not water but count as "water")?
- According to the histogram, where is it best to place the threshold? Does the same threshold work for both dates?
- Why are there peaks exactly at -1 and at 1?

> Spoiler: choosing the "perfect" threshold is more art than science. But don't say it out loud in front of someone who does supervised classification.


In [ ]:
# NDWI of each date, histograms and possible water masks (NDWI > threshold: water)
# .............................
# Complete the missing code:
# .............................
threshold_1 = 'COMPLETE THE CODE HERE'
threshold_2 = 'COMPLETE THE CODE HERE'

ndwi = {}
for date in dates:
    B03, B08 = ndwi_bands[date]["B03"], ndwi_bands[date]["B08"]
    band_sum = B03 + B08
    ndwi[date] = np.divide(B03 - B08, band_sum, out=np.full_like(band_sum, np.nan), where=band_sum > 0)
    print(f"{date}: NDWI minimum = {np.nanmin(ndwi[date]):.2f} | maximum = {np.nanmax(ndwi[date]):.2f}")

# 1) NDWI histograms (logarithmic vertical axis: there are far fewer water pixels than land pixels)
fig, ax = plt.subplots(figsize=(9, 4))
for date in dates:
    values = ndwi[date][~np.isnan(ndwi[date])]   # without the NaN
    ax.hist(values, bins=200, alpha=0.6, label=date)
ax.axvline(threshold_1, color="black", linestyle="--", label=f"threshold 1 = {threshold_1}")
ax.axvline(threshold_2, color="black", linestyle=":", label=f"threshold 2 = {threshold_2}")
ax.set_yscale("log")
ax.set_xlabel("NDWI")
ax.set_ylabel("Number of pixels")
ax.legend()
plt.show()

# 2) 3x2 grid: NDWI (row 1) and NDWI masked with each threshold, water only (rows 2 and 3)
fig, axs = plt.subplots(3, 2, figsize=(12, 13))
for col, date in enumerate(dates):
    im = axs[0, col].imshow(ndwi[date], cmap="RdBu", vmin=-1, vmax=1)
    axs[0, col].set_title(f"NDWI: {date}")
    fig.colorbar(im, ax=axs[0, col], label="NDWI", shrink=0.8)

    for row, threshold in enumerate([threshold_1, threshold_2], start=1):
        water = apply_mask(ndwi[date], threshold)
        print(f"{date}, NDWI > {threshold}: {water.sum()} water pixels ({water.mean():.1%} of the image)")

        im = axs[row, col].imshow(np.where(water, ndwi[date], np.nan), cmap="RdBu", vmin=-1, vmax=1)   # NaN: not drawn
        axs[row, col].set_title(f"Masked NDWI (NDWI > {threshold}): {date}")
        fig.colorbar(im, ax=axs[row, col], label="NDWI", shrink=0.8)

for ax in axs.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

<details>
<summary><b>Show answers</b> (try to answer before opening)</summary>

**1. What happens when you lower or raise the threshold value?**

**Raising** the threshold shrinks the mask: only the pixels with the highest NDWI remain. **Lowering** it enlarges the mask and pixels with NDWI close to 0 come in. But the reservoir barely notices: 95 % of its pixels have an NDWI of about 0.3 or more, and more than half are exactly 1, far from both thresholds. What changes are the edges:

- **2017:** going from 0 to 0.1 removes 308 pixels (3 % of the mask). Two thirds are on the shore of the reservoir: they are **mixed pixels**, which mix water and land within the 15 m × 15 m of the pixel.
- **2024:** the same change removes 1,519 pixels (27 % of the mask), and almost all of them (1,346) are **cloud edges**.

If you lower the threshold below 0 (for example, to -0.1), even more pixels that are not water come in: in 2024, about 2,800 more, almost all of them clouds.

**2. Are there misclassified areas?**

Yes, especially in 2024:

- **Clouds:** they reflect a lot, and similarly, in the green and in the infrared, so their NDWI stays close to 0. With the threshold at 0, their edges come in as "water". They can be seen at the top left and on the right of the image, over the clouds.
- **Shores:** mixed pixels have an intermediate NDWI and enter or leave the mask depending on the threshold. This is not an error of the index, but of the resolution: the real edge of the water runs through the pixel.
- **Scattered spots:** a few small spots of a few pixels remain, far from the reservoir. They may be small ponds or noise; at this resolution it cannot be told for sure.

Cloud **shadows**, on the other hand, are generally not confused with water: they are dark in both bands, but especially in the green, and their NDWI is very negative.

**3. Where is it best to place the threshold? Does the same one work for both dates?**

The histogram shows two groups: a large peak around -0.7 (land) and a tail of high values (water). Between the two, roughly from 0 to 0.3, there are very few pixels: any threshold in that **valley** separates the reservoir almost the same way. The difference between the dates lies in the clouds: in 2024 they form a small bump right around 0, which does not appear in 2017. That is why 0.1 works well for both dates, and 0 works in 2017 but not in 2024: **the threshold depends on each image**, and it is a good idea to check it with its histogram. To compare dates, ideally use the **same threshold** that falls in the valley of all of them: that way the measured change is due to the reservoir and not to having changed the criterion. If the images are very different (turbid water, another season, haze), one per date may be needed.

> There are methods that choose the threshold automatically from the histogram, such as **Otsu's method**, which looks for the cut that best separates the two groups.

**4. Why are there peaks exactly at -1 and at 1?**

They are pixels in which **one of the two bands is 0**:

- **NDWI = 1:** the infrared (B08) is 0. This happens in deep water, which absorbs almost all the infrared. They are more than half of the reservoir pixels.
- **NDWI = -1:** the green (B03) is 0. This happens in some very dark pixels, such as shadows.

If B08 = 0, the formula gives $\frac{B03 - 0}{B03 + 0} = 1$; and if B03 = 0, it gives $\frac{0 - B08}{0 + B08} = -1$. That is why those pixels pile up at the exact ends of the range.

</details>


#### Image Subtraction for Change Detection

In remote sensing, comparing images from different moments is a powerful way to detect changes over time and quantify their magnitude (e.g., in lakes, reservoirs or other water surfaces, vegetation, etc.).

If you have the NDWI for two dates (for example, a wet year and a dry year), you can subtract:

$$\text{Change} = \text{NDWI}{(\text{later date})} - \text{NDWI}{(\text{earlier date})}$$

- **Positive** values indicate an increase in the NDWI (possible increase in water).
- **Negative** values indicate a decrease (possible retreat of the water body).

The resulting image highlights the areas where the NDWI changed. To measure how much the reservoir shrank or expanded, it is also worth comparing the **water masks** of both dates.

**Exercises:**

1. Choose a threshold from the previous analysis (histograms and masks) and store it in the variable `chosen_threshold`.
2. Calculate the NDWI change between the two dates and show it with a diverging colormap (for example, `'RdBu'`) and a symmetric scale, from -2 to 2: the possible range of an NDWI difference. What happens if you turn off the scale?
3. Subtract the water masks of the two dates, converted to integers (`True` = 1, `False` = 0), and show the result: -1 is lost water, 0 is no change and +1 is new water.
4. Interpret the water surface of each date in hectares. The pixel size is in degrees (EPSG:4326, in `meta["transform"]`): to convert it to meters, 1° of latitude ≈ 110.5 km and 1° of longitude ≈ 111.3 km × cos(latitude).

**Questions:**

- Are all the blue areas of the NDWI change map new water? What else can make the NDWI go up?
- What does the mask difference show that the NDWI change map does not? And the other way around?
- How much did the reservoir shrink between the two dates? How much would you trust that number?

In [ ]:
# Change between dates: NDWI and water masks
# .............................
# Complete the missing code:
# .............................
# 1. Threshold chosen from the previous analysis
chosen_threshold = 'COMPLETE THE CODE HERE'

early, late = dates
# .............................
# Complete the missing code:
# .............................
# 2. NDWI change between the two dates
change = 'COMPLETE THE CODE HERE'

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
im = axs[0].imshow(change, cmap="RdBu", vmin=-2, vmax=2)   # possible range of an NDWI difference
axs[0].set_title(f"NDWI change: {late} − {early}")
fig.colorbar(im, ax=axs[0], label="NDWI change", shrink=0.8)

# .............................
# Complete the missing code:
# .............................
# 3. Subtract the masks (True = 1, False = 0)
water_early = 'COMPLETE THE CODE HERE'
water_late = 'COMPLETE THE CODE HERE'
water_change = 'COMPLETE THE CODE HERE'   # -1: lost water, 0: no change, +1: new water

im = axs[1].imshow(water_change, cmap="RdBu", vmin=-1, vmax=1)
axs[1].set_title(f"Water mask difference (NDWI > {chosen_threshold})")
cbar = fig.colorbar(im, ax=axs[1], ticks=[-1, 0, 1], shrink=0.8)
cbar.ax.set_yticklabels(["lost water", "no change", "new water"])
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

# 4. Area: the pixel size is in degrees (EPSG:4326).
# 1° of latitude ≈ 110.5 km and 1° of longitude ≈ 111.3 km × cos(latitude)
t = meta["transform"]
latitude = t.f + t.e * meta["height"] / 2   # latitude of the center of the image
ha_pixel = (t.a * 111_320 * np.cos(np.radians(latitude))) * (-t.e * 110_540) / 10_000

print(f"Area of one pixel: {ha_pixel * 10_000:.0f} m²")
print(f"Water on {early}: {water_early.sum() * ha_pixel:.0f} ha")
print(f"Water on {late}: {water_late.sum() * ha_pixel:.0f} ha")
print(f"Lost water: {(water_change == -1).sum() * ha_pixel:.0f} ha | new water: {(water_change == 1).sum() * ha_pixel:.0f} ha")

<details>
<summary><b>Show answers</b> (try to answer before opening)</summary>

**1. Are all the blue areas of the change map new water?**

No. The large blue spots, on the right and at the top left of the image, are the **clouds** of 2024. The NDWI of a cloud stays close to 0, while that of the land it covers is around -0.7: the difference gives an increase of between 0.5 and 0.8, even though there is not a single drop of new water. Of the pixels whose NDWI rises by more than 0.5, 96 % are clouds. In contrast, where the reservoir retreated the NDWI drops a lot (around -1.2, in red), and on cloud-free land it barely changes (close to 0).

That is why the NDWI change map is useful to see **where** something changed, but it is not enough to say **what** changed: it has to be interpreted together with the images of each date.

**What if you turn off the scale?** Without `vmin` or `vmax`, the scale goes from the minimum (-1.69) to the maximum (+1.23) of the change, and white, the center of the colormap, falls at -0.23 instead of 0. 76 % of the pixels barely change (less than 0.1), but they turn **light blue**, and the whole image looks like an increase in water. With the symmetric scale, from -2 to 2, white stays at 0 and means "no change": it is the same thing we saw with the NDVI in Step 6.

**2. What does the mask difference show that the NDWI change map does not? And the other way around?**

- **The mask difference** answers directly where there is lost or new water: each pixel is -1, 0 or +1. Clouds do not appear as new water, because their NDWI does not exceed the threshold on either date. And since each pixel is either water or not, the area can be counted in hectares.
- **The NDWI change map** shows how much each pixel changed, also outside the water: clouds and changes on land can be seen there. But it does not say what is water and what is not.

The mask **simplifies** (it decides water yes or no) and that is why it is useful for measuring; the NDWI **keeps the detail** and that is why it is useful for interpreting. They are used together.

**3. How much did the reservoir shrink? How much would you trust that number?**

With a threshold of 0.1, the water goes from about **218 ha** on 6/4/2017 to about **88 ha** on 4/8/2024: the reservoir lost around **60 %** of its surface (131 ha of lost water and only 1 ha of new water).

It is a good estimate, but not an exact one:

- **It depends on the threshold:** with 0 instead of 0.1, it gives 225 ha and 121 ha, because in 2024 the cloud edges come in.
- **Mixed pixels:** each pixel is about 15 m × 15 m (≈ 220 m²), and on the shore it mixes water and land. With a narrow reservoir, like the one in 2024, the shore weighs a lot in the total.
- **Conversion from degrees to meters:** it is approximate, and uses the latitude of the center of the image.
- **Only two dates:** one in spring and one in summer. They show how much the reservoir changed between those two days, but are not enough to know whether it is a trend or a seasonal variation.

</details>


### Step 9: NDVI Analysis for Multiple Files (time series)

In this part we are going to study techniques to work with several images. When the number of images grows, looking at them one by one stops being practical: we need mathematical tools that summarize each image in a few numbers and show how they change over time. We will study the evolution of the vegetation in an area of Barcelona (my neighborhood: part of Montjuïc, Sants and the Zona Franca) between 2015 and 2026, with one image per year.

But first, let's go back a few steps:

1. **Metrics with masks.** Let's look at a small example; a 3 × 4 matrix, a miniature NDVI. In Step 8 we used masks to **draw** only the water; now we will use them to summarize each image in a few numbers, for example, the percentage of pixels with vegetation (NDVI above a threshold).
    - Define a threshold (`threshold`) and create a mask (`mask`) with the pixels that exceed it.
    - Use the mask to select those values (`my_ndvi[mask]`) and calculate their sum and mean. What shape does the result have?
    - Calculate how many pixels exceed the threshold and what percentage of the total they represent.

    What is the difference between `my_ndvi[mask]` and `np.where(mask, my_ndvi, np.nan)`, which we used in Step 8? And between `mask.sum()` and `my_ndvi[mask].sum()`?

In [ ]:
my_ndvi = np.array([[0.15, 0.05, 0.30, 0.20],
                    [0.10, 0.35, 0.20, 0.20],
                    [0.01, 0.50, 0.20, 0.20]])
print('NDVI:\n', my_ndvi)

# Mask over a matrix: select values and calculate metrics
# .............................
# Complete the missing code:
# .............................
threshold = 'COMPLETE THE CODE HERE'
mask = 'COMPLETE THE CODE HERE'    # same shape as my_ndvi: (3, 4)

print('Mask:\n', mask)

# .............................
# Complete the missing code:
# .............................
values = 'COMPLETE THE CODE HERE'    # only the values where the mask is True: 1-dimensional vector

print('Selected values:', values, '| shape:', values.shape)
print(f'Sum of the values: {values.sum():.2f} | mean: {values.mean():.3f}')

# .............................
# Complete the missing code:
# .............................
n_pixels = 'COMPLETE THE CODE HERE'     # each True counts as 1
percentage = 'COMPLETE THE CODE HERE'   # proportion of True, in %

print('Pixels above the threshold:', n_pixels)
print(f'Percentage of the total: {percentage:.1f} %')

# To draw, it is better to keep the shape (Step 8): np.where puts NaN where the mask is False
print('With np.where:\n', np.where(mask, my_ndvi, np.nan))

<details>
<summary><b>Show answers</b> (try to answer before opening)</summary>

**What is the difference between `my_ndvi[mask]` and `np.where(mask, my_ndvi, np.nan)`?**

- `my_ndvi[mask]` **selects** only the values where the mask is `True` and returns them in a 1-dimensional vector: `[0.3, 0.35, 0.5]`, with shape `(3,)`. The location of each pixel is lost, but it is exactly what is needed to **calculate** (mean, sum, count).
- `np.where(mask, my_ndvi, np.nan)` **keeps the shape** of the matrix, `(3, 4)`, and puts `NaN` where the mask is `False`. Each value stays in its place, so it is useful to **draw** a map, as in Step 8.

**And between `mask.sum()` and `my_ndvi[mask].sum()`?**

- `mask.sum()` **counts** how many pixels exceed the threshold, because each `True` is worth 1: it gives **3**.
- `my_ndvi[mask].sum()` **adds up the NDVI values** of those pixels: 0.3 + 0.35 + 0.5 = **1.15**.

For the percentage of vegetation the first one is used: `mask.mean() * 100` = 3 of 12 pixels = **25 %**.

</details>

2. **Time series of Barcelona (Montjuïc, Sants and the Zona Franca), from 2015 to 2026.** There is one image per year, always from **July or August**: the one with the fewest clouds within the area. Choosing them at the same time of year prevents the differences between years from being due to the season (Mediterranean vegetation changes a lot between spring and the end of summer). The images are in `raw/time_series_sants/`.

    The next cells reuse what you did in the previous steps:
    - They load bands B04 and B08 of each year with `read_band`, calculate the NDVI (as in Step 6) and show it in a grid of 4 rows and 3 columns, with the same fixed scale (from -1 to 1) for all years.
    - With a vegetation mask (NDVI > `veg_threshold`) and the `apply_mask` function from Step 8, they calculate for each year the **mean NDVI** of the whole image, the **percentage of vegetation** and the **mean NDVI of the vegetation**, store them in a table (`DataFrame`) and plot them as a function of the year.

    Look at the grid and the charts: which years look greener? Do they match what the metrics show?

In [ ]:
# NDVI of each year and 4x3 grid
directory_path = Path('raw/time_series_sants')
annual_ndvi = {}   # {date: NDVI}
for file_path in sorted(directory_path.glob("*B04_(Raw).tiff")):
    date = re.search(r"\d{4}-\d{2}-\d{2}", file_path.name).group()
    B04, meta = read_band(file_path)
    B08, _ = read_band(directory_path / file_path.name.replace("B04", "B08"))
    band_sum = B04 + B08
    annual_ndvi[date] = np.divide(B08 - B04, band_sum, out=np.full_like(band_sum, np.nan), where=band_sum > 0)
series_dates = sorted(annual_ndvi)
print(len(series_dates), "images:", series_dates)

fig, axs = plt.subplots(4, 3, figsize=(13, 10))
for ax, date in zip(axs.ravel(), series_dates):
    im = ax.imshow(annual_ndvi[date], cmap="RdYlGn", vmin=-1, vmax=1)   # same scale for all years
    ax.set_title(date)
for ax in axs.ravel():
    ax.axis("off")
fig.colorbar(im, ax=axs, label="NDVI", shrink=0.6)
plt.show()

In [ ]:
# Metrics of each year with the vegetation mask
import pandas as pd

veg_threshold = 0.4   # NDVI > veg_threshold: vegetation

rows = []
for date in series_dates:
    ndvi = annual_ndvi[date]
    vegetation = apply_mask(ndvi, veg_threshold)   # function from Step 8
    rows.append({
        "year": int(date[:4]),
        "mean NDVI": np.nanmean(ndvi),
        "% vegetation": vegetation.mean() * 100,
        "mean NDVI of vegetation": ndvi[vegetation].mean(),
    })
metrics = pd.DataFrame(rows).set_index("year")
print(metrics.round(3))

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
metrics[["mean NDVI", "mean NDVI of vegetation"]].plot(ax=axs[0], marker="o")
axs[0].set_ylabel("NDVI")
metrics["% vegetation"].plot(ax=axs[1], marker="o", color="tab:green")
axs[1].set_ylabel(f"% of pixels with NDVI > {veg_threshold}")
for ax in axs:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

3. **Something new: the trend of each pixel.** The metrics summarize the whole image in one number per year, but they don't say *where* the vegetation changed. For that, the next cell fits a straight line to the NDVI of **each pixel** over the years (`np.polyfit`) and shows a map of its **slope**: how much the NDVI goes up or down per year. Red spots are areas that lost vegetation steadily; green ones, areas that gained it. Before the map, a chart shows the series of three example pixels with their line, to see what the slope measures.

To find out what happened in an area, look it up in **Google Maps**. The file's georeferencing (`meta["transform"]`, which you already used in Step 8) converts a position in the image (column, row) into coordinates (longitude, latitude): `lon, lat = meta["transform"] * (col + 0.5, row + 0.5)`. The `+ 0.5` points to the center of the pixel. Copy `lat, lon` into Google Maps (satellite view) to see what is there today.

Key questions:

- How did the vegetation evolve between 2015 and 2026? Is there any unusual year?
- Which of the three metrics changes most from one year to the next? Why does the mean NDVI of the vegetation barely change?
- What can explain the red and green areas of the trend map? Look up the marked areas in Google Maps (the cell prints their coordinates): are they all permanent changes?
- Look at the series of the example pixels: is a straight line a good model for all changes? What does the slope say when the change is sudden?

In [ ]:
# Trend of the NDVI in each pixel: slope of a straight line fitted over the years
years = np.array([int(d[:4]) for d in series_dates])
stack = np.stack([annual_ndvi[d] for d in series_dates])   # (years, rows, columns)
stack = np.where(np.isnan(stack), np.nanmean(stack, axis=0), stack)   # the few NaN are filled with the pixel mean

# np.polyfit fits all the columns at once: each column is the series of one pixel
slope = np.polyfit(years, stack.reshape(len(years), -1), 1)[0].reshape(stack.shape[1:])

# Three example pixels (row, column): their NDVI of each year and the fitted line
examples = {
    "stable (Montjuïc)": (48, 361),
    "sudden drop (Zona Franca)": (215, 90),
    "new park (Gran Via and Amadeu Oller)": (50, 200),
}
colors = ["tab:green", "tab:purple", "tab:blue"]   # no red: on the map, red is NDVI loss
fig, ax = plt.subplots(figsize=(12, 4))
for (name, (row, col)), color in zip(examples.items(), colors):
    series = stack[:, row, col]
    m, b = np.polyfit(years, series, 1)   # slope and intercept of the line
    ax.plot(years, series, "o", color=color, label=f"{name}:\nslope {m:+.3f} per year")
    ax.plot(years, m * years + b, "--", color=color)
ax.set_xlabel("year")
ax.set_ylabel("NDVI")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))   # legend outside the chart, so it does not cover points
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

lim = np.nanpercentile(np.abs(slope), 99)   # symmetric scale: white is "no change"
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
im = axs[0].imshow(np.nanmean(stack, axis=0), cmap="RdYlGn", vmin=-1, vmax=1)
axs[0].set_title(f"Mean NDVI {years[0]}–{years[-1]}")
fig.colorbar(im, ax=axs[0], label="NDVI", shrink=0.8)
im = axs[1].imshow(slope, cmap="RdYlGn", vmin=-lim, vmax=lim)
axs[1].set_title("NDVI trend")
fig.colorbar(im, ax=axs[1], label="NDVI change per year", shrink=0.8)
for (name, (row, col)), color in zip(examples.items(), colors):   # location of the example pixels
    axs[1].plot(col, row, "o", markersize=12, markerfacecolor="none", markeredgecolor=color, markeredgewidth=2)

# Other areas with NDVI loss (row, column), marked with a square
other_areas = {
    "swimming club": (128, 229),
    "stadium": (94, 336),
}
for name, (row, col) in other_areas.items():
    axs[1].plot(col, row, "s", markersize=14, markerfacecolor="none", markeredgecolor="black", markeredgewidth=2)
    axs[1].annotate(name, (col, row), xytext=(12, 12), textcoords="offset points", fontsize=9,
                     bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))   # white background so it can be read
for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Coordinates of the marked areas, to look them up in Google Maps (latitude, longitude)
for name, (row, col) in {**examples, **other_areas}.items():
    lon, lat = meta["transform"] * (col + 0.5, row + 0.5)   # center of the pixel
    print(f"{name}: {lat:.5f}, {lon:.5f}")

<details>
<summary><b>Show answers</b> (try to answer before opening)</summary>

**1. How did the vegetation evolve between 2015 and 2026? Is there any unusual year?**

The percentage of vegetation (NDVI > 0.4) rises from 18 % in 2015 to **26 % in 2020**, drops to **18 % in 2023** and recovers to 23 % in 2025–2026. The unusual year is **2023**: it has the least vegetation of the series and coincides with the worst of the **drought in Catalonia** (2021–2024). 2020, on the other hand, is the greenest. These are year-to-year changes, not a trend: summer vegetation depends a lot on that year's rainfall.

**2. Which of the three metrics changes most? Why does the mean NDVI of the vegetation barely change?**

- **The percentage of vegetation** is the one that changes most: it goes from 18 % to 26 %.
- **The mean NDVI of the whole image** changes little (from 0.19 to 0.26), because most of the area is city, with low NDVI, and the city barely changes.
- **The mean NDVI of the vegetation** also changes little (from 0.56 to 0.62), because of how the mask works: in a dry year, the pixels that dry out fall below the threshold and **leave the mask**, so the mean is calculated only with the vegetation that is still green. The drought shows in *how much* area exceeds the threshold, not in the NDVI of the area that exceeds it.

**3. What can explain the red and green areas of the map? Are they all permanent changes?**

No. The map shows **where** the NDVI changed, but the **cause** has to be investigated. Looking up the marked areas in Google Maps:

- **New park (Gran Via / Amadeu Oller):** it is a park behind a supermarket. The NDVI rises from almost 0 to 0.3–0.4 from 2020 onward, as the vegetation grows. It is a **real and permanent change**: green was gained.
- **Swimming club (west of Montjuïc):** the pools are still there. What changes is the lawn around them: the NDVI drops from 0.5 to **0.06 in 2023** and partly recovers (0.25). Everything points to an effect of the drought and of irrigation restrictions, rather than a change of use.
- **Stadium:** the stadium did not change. The NDVI of the pitch is high almost every year, but in 2025 and 2026 it is negative: on the day of the image the grass was probably **covered** (for example, by a stage) or being replaced. It is a **temporary change** that the image caught on that very day.
- **Sudden drop (Zona Franca):** the NDVI goes from 0.4 to almost 0 in 2023, but the Google Maps photos always show the same thing. It may be irrigated vegetation that dried out in the drought and did not recover, but **with these data it cannot be confirmed**.

Not everything you get is clear at first sight. With **a single image per year**, a one-day change (a covered pitch) or a seasonal one (an unwatered lawn) is confused with a permanent change. To understand it better it has to be studied with **more images**: several per year, or from other times of year.

**4. Is a straight line a good model for all changes?**

Not for all of them. In the **new park**, the NDVI rises gradually and the line follows it well. In the **Zona Franca**, the NDVI stays at 0.4 until 2022 and drops suddenly in 2023: the line (-0.040 per year) shows it as a steady drop since 2015, which never existed. The slope is useful to **find** where something changed, but to know **when** and **how** you have to look at the pixel's series.

</details>